# 🌲 Random Forests
**ISLP Chapter 8 · Data Pattern Recognition for the Rest of Us**

> Random Forests fix the main weakness of single decision trees — high variance — by averaging hundreds of trees, each trained on a random subset of data and features.

### Dataset
**IBM HR Analytics — Employee Attrition**
Predict which employees are likely to leave the company. Real HR data with 35 features including salary, job satisfaction, overtime, and years at company. Directly applicable to workforce planning and retention strategy.

### Time: ~60 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.preprocessing import LabelEncoder
print("\u2713 Setup complete")

In [ ]:
# IBM HR Analytics — Employee Attrition dataset
# Source: IBM Watson Analytics sample dataset (widely used in HR analytics)
import numpy as np, pandas as pd

try:
    url = 'https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv'
    attrition = pd.read_csv(url)
    print("Loaded IBM HR dataset from source")
except:
    # Reconstruct synthetically with realistic HR properties
    np.random.seed(42)
    n = 1470
    age = np.random.randint(18, 60, n)
    monthly_income = np.random.lognormal(8.5, 0.5, n).astype(int).clip(1000, 20000)
    job_satisfaction = np.random.randint(1, 5, n)
    overtime = np.random.binomial(1, 0.28, n)
    years_at_company = np.random.randint(0, 35, n)
    work_life_balance = np.random.randint(1, 5, n)
    distance_from_home = np.random.randint(1, 30, n)
    num_companies_worked = np.random.randint(0, 10, n)
    department = np.random.choice(['Sales','R&D','HR'], n, p=[0.3, 0.6, 0.1])

    # Attrition driven by: overtime, low satisfaction, low income, early career
    log_odds = (-1.5
                + 1.2 * overtime
                - 0.3 * job_satisfaction
                - 0.0001 * monthly_income
                + 0.3 * (years_at_company < 3).astype(float)
                + 0.1 * distance_from_home / 10
                + 0.2 * num_companies_worked / 5)
    prob_leave = 1 / (1 + np.exp(-log_odds))
    attrition_flag = (np.random.uniform(0,1,n) < prob_leave).astype(int)

    attrition = pd.DataFrame({
        'Age': age, 'MonthlyIncome': monthly_income,
        'JobSatisfaction': job_satisfaction, 'OverTime': np.where(overtime,'Yes','No'),
        'YearsAtCompany': years_at_company, 'WorkLifeBalance': work_life_balance,
        'DistanceFromHome': distance_from_home, 'NumCompaniesWorked': num_companies_worked,
        'Department': department, 'Attrition': np.where(attrition_flag,'Yes','No')
    })
    print("Using synthetic IBM HR-like dataset")

# Encode target and categoricals
attrition['Attrition_num'] = (attrition['Attrition'] == 'Yes').astype(int)
cat_cols = attrition.select_dtypes('object').columns.drop('Attrition')
for c in cat_cols:
    attrition[c] = LabelEncoder().fit_transform(attrition[c].astype(str))

feature_cols = [c for c in attrition.columns if c not in ['Attrition','Attrition_num']]
X = attrition[feature_cols]
y = attrition['Attrition_num']

print(f"\nIBM HR Attrition: {attrition.shape[0]:,} employees, {len(feature_cols)} features")
print(f"Attrition rate: {y.mean():.1%}  (class imbalance — typical for HR data)")
print(f"\nTop features: {feature_cols[:8]}")

## 🌳 Part 1 — Why Random Forests Beat Single Trees

A single decision tree memorizes the training data (100% training accuracy) but generalizes poorly.
Random Forests train hundreds of trees on bootstrap samples, each considering only a random
subset of features at each split. The decorrelation between trees is what makes the average powerful.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                             random_state=42, stratify=y)

# Compare single tree vs random forest
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_tr, y_tr)

rf = RandomForestClassifier(n_estimators=500, oob_score=True,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

print(f"{'Metric':<28} {'Decision Tree':>14} {'Random Forest':>14}")
print("-" * 58)
for metric, t_val, rf_val in [
    ('Training accuracy',     tree.score(X_tr,y_tr),   rf.score(X_tr,y_tr)),
    ('Test accuracy',         tree.score(X_te,y_te),   rf.score(X_te,y_te)),
    ('Test AUC-ROC',          roc_auc_score(y_te, tree.predict_proba(X_te)[:,1]),
                              roc_auc_score(y_te, rf.predict_proba(X_te)[:,1])),
    ('Generalization gap',    tree.score(X_tr,y_tr)-tree.score(X_te,y_te),
                              rf.score(X_tr,y_tr)-rf.score(X_te,y_te))]:
    print(f"  {metric:<26} {t_val:>14.3f} {rf_val:>14.3f}")
print(f"\n  OOB error (free CV estimate):  {'N/A':>14} {1-rf.oob_score_:>14.3f}")

In [ ]:
# How test error changes with number of trees
n_trees_range = [1, 5, 10, 25, 50, 100, 200, 500]
oob_errors, test_aucs = [], []

for n in n_trees_range:
    rf_n = RandomForestClassifier(n_estimators=n, oob_score=True,
                                   class_weight='balanced', random_state=42, n_jobs=-1)
    rf_n.fit(X_tr, y_tr)
    oob_errors.append(1 - rf_n.oob_score_)
    test_aucs.append(roc_auc_score(y_te, rf_n.predict_proba(X_te)[:,1]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(n_trees_range, oob_errors, 's--', color='#e85d20', lw=2, label='OOB error')
axes[0].set_xlabel('Number of Trees'); axes[0].set_ylabel('Error Rate')
axes[0].set_title('OOB Error Stabilizes Around 100-200 Trees')
axes[0].set_xscale('log'); axes[0].legend()

axes[1].plot(n_trees_range, test_aucs, 'o-', color='#1e5fa8', lw=2, label='Test AUC-ROC')
axes[1].set_xlabel('Number of Trees'); axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('AUC-ROC vs Number of Trees\n(attrition prediction — IBM HR data)')
axes[1].set_xscale('log'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Which factors drive employee attrition? — Variable Importance
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#e85d20' if v > importances.quantile(0.75) else '#1e5fa8'
          for v in importances]
ax.barh(importances.index, importances.values, color=colors, edgecolor='white')
ax.set_xlabel('Mean Decrease in Gini Impurity')
ax.set_title('What Drives Employee Attrition?\n'
             'Random Forest Variable Importance (orange = top quartile)')
plt.tight_layout(); plt.show()

top3 = importances.tail(3).index.tolist()[::-1]
print(f"\n\u2714 Top 3 attrition drivers: {', '.join(top3)}")
print("  These are the levers HR should focus on for retention strategy")

In [ ]:
# ROC curve + business interpretation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

RocCurveDisplay.from_estimator(rf, X_te, y_te, ax=axes[0],
    name=f'Random Forest (AUC={roc_auc_score(y_te,rf.predict_proba(X_te)[:,1]):.3f})')
RocCurveDisplay.from_estimator(tree, X_te, y_te, ax=axes[0],
    name='Single Tree', color='#e85d20', alpha=0.7)
axes[0].set_title('ROC Curve — Attrition Prediction')

# Score distribution by actual outcome
rf_scores = rf.predict_proba(X_te)[:,1]
axes[1].hist(rf_scores[y_te==0], bins=30, alpha=0.6, color='#1e5fa8',
             density=True, label='Stayed')
axes[1].hist(rf_scores[y_te==1], bins=30, alpha=0.6, color='#e85d20',
             density=True, label='Left')
axes[1].axvline(0.5, color='black', lw=1.5, ls='--', label='Default threshold')
axes[1].set_xlabel('Predicted Attrition Probability')
axes[1].set_title('Score Distribution\n(good separation = useful model)')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
answers = {
    "q1": "",  # What is the key difference between bagging and Random Forests?
    "q2": "",  # Why does OOB error give a valid estimate of test error?
    "q3": "",  # What does variable importance measure in a Random Forest?
    "q4": "",  # If HR wants to reduce attrition, which top feature would you investigate first and why?
    "q5": "",  # Why does a single decision tree have 100% training accuracy but Random Forest does not?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("\u274c Still empty: "+str(missing) if missing else "\u2705 Done! File \u2192 Save a copy in GitHub")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Random Forests"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*